In [2]:

# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


from langchain import hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma, FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

USER_AGENT environment variable not set, consider setting it to identify your requests.


# 데이터 로드 및 전처리 

In [ ]:
import json
import pandas as pd
from datetime import datetime

with open ("instagram_diary/content/posts_1.json", "r") as f:
    data = json.load(f)

# DataFrame 생성
df = pd.DataFrame()
for i in range(len(data)):
    df = pd.concat([df,pd.DataFrame(data[i])])

# 유니코드 이스케이프 시퀀스를 디코딩
df = df.map(lambda x: x.encode('latin-1').decode('utf-8') if isinstance(x, str) else x)
df = df.reset_index(drop=True)

#  전처리 
df.loc[df.title.isna(),'creation_timestamp']= [x.get('creation_timestamp') for x in df.loc[df.title.isna()]['media']]
df.loc[df.title.isna(),'title']= [x.get('title').encode('latin-1').decode('utf-8') for x in df.loc[df.title.isna()]['media']]
df['dt'] = [datetime.fromtimestamp(x) for x in df['creation_timestamp']]
df['year'] = df['dt'].map(lambda x: x.year)
df['month'] = df['dt'].map(lambda x: x.month)
df['day'] = df['dt'].map(lambda x: x.day)

# 결과 출력
print(df.shape)
# df.head()

# 중복 제거
df_nodup = df[["title", 'year', 'month', 'day']].drop_duplicates(keep='last')
print(df_nodup.shape)
df_nodup["title"] = df_nodup.title.map(lambda x: x.replace('오빠', '김땡땡이').replace('유환', '땡땡').replace('스티븐', '탕탕') )
df_nodup.reset_index(drop=True).head()

In [28]:
import re 
tmp = '워니, 소니'
print(['워니', '소니'] in tmp)
# tmp2 = re.sub(r" ", "", tmp).split(',')
# tmp2

TypeError: 'in <string>' requires string as left operand, not list

In [3]:
# metadata 버림 


import json
import pandas as pd
from datetime import datetime

with open ("instagram_diary/content/posts_1.json", "r") as f:
    data = json.load(f)

# DataFrame 생성
df = pd.DataFrame()
for i in range(len(data)):
    df = pd.concat([df,pd.DataFrame(data[i])])

# 유니코드 이스케이프 시퀀스를 디코딩
df = df.map(lambda x: x.encode('latin-1').decode('utf-8') if isinstance(x, str) else x)
df = df.reset_index(drop=True)

#  전처리 
df.loc[df.title.isna(),'creation_timestamp']= [x.get('creation_timestamp') for x in df.loc[df.title.isna()]['media']]
df.loc[df.title.isna(),'title']= [x.get('title').encode('latin-1').decode('utf-8') for x in df.loc[df.title.isna()]['media']]
df['dt'] = [datetime.fromtimestamp(x) for x in df['creation_timestamp']]
df['year'] = df['dt'].map(lambda x: x.year)
df['month'] = df['dt'].map(lambda x: x.month)
df['day'] = df['dt'].map(lambda x: x.day)
df['combined'] = df['year'].map(str) + "년" + df['month'].map(str) + '월' + df['day'].map(str) + '일 일기입니다. \n' + df['title'].map(str)


# 결과 출력
print(df.shape)
# df.head()

# 중복 제거
df_nodup = df[["combined"]].drop_duplicates(keep='last')
print(df_nodup.shape)
df_nodup["combined"] = df_nodup.combined.map(lambda x: x.replace('오빠', '김땡땡이').replace('유환', '땡땡').replace('스티븐', '탕탕') )
df_nodup.reset_index(drop=True).head()

(4455, 8)
(1332, 1)


,combined
0,2024년7월15일 일기입니다. \n인도네시아로 떠나버린 엄마와 영종도로 떠나버린 ...
1,2024년7월13일 일기입니다. \n아니 오늘 사진 이거 밖에 없어? 근데 오늘 정...
2,2024년7월13일 일기입니다. \n퇴근하구 김땡땡이랑 양재에서 술막엇다. 출근해서...
3,2024년7월11일 일기입니다. \n오늘 요약 재밋엇음 은근 사람만나는것도 재밋네 ...
4,2024년7월11일 일기입니다. \nㅎㅏ 별개로 누님과의 만남은 즐거웠움 김땡땡 호...


In [4]:
# df_nodup.title.map(lambda x: x.find('현진')).value_counts()
df_nodup.title.map(lambda x: x.find('꾸')).value_counts()

AttributeError: 'DataFrame' object has no attribute 'title'

In [5]:
# 단계 1: 문서 로드(Load Documents)
# 문서를 로드하고, 청크로 나누고, 인덱싱합니다.
from langchain_community.document_loaders import DataFrameLoader
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_community.vectorstores import FAISS


# PDF 파일 로드. 파일의 경로 입력
loader = DataFrameLoader(df_nodup, page_content_column="combined")

# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=50)

split_docs = loader.load_and_split(text_splitter=text_splitter)

# 단계 3, 4: 임베딩 & 벡터스토어 생성(Create Vectorstore)
# 벡터스토어를 생성합니다.
# vectorstore = FAISS.from_documents(documents=split_docs, embedding=OpenAIEmbeddings())
vectorstore = FAISS.from_documents(documents=split_docs, embedding=HuggingFaceBgeEmbeddings())
# vectorstore = FAISS.from_documents(documents=split_docs, embedding=FastEmbedEmbeddings())

# 단계 5: 리트리버 생성(Create Retriever)
# 사용자의 질문(query) 에 부합하는 문서를 검색합니다.


/opt/anaconda3/envs/insta/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [6]:

# 유사도 높은 K 개의 문서를 검색합니다.
k = 6

# (Sparse) bm25 retriever and (Dense) faiss retriever 를 초기화 합니다.
bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = k
bm25_retriever.metadata = {"year":'2021'}

# faiss_vectorstore = FAISS.from_documents(split_docs, OpenAIEmbeddings())
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": k, 'filter': {'year':'2021'}})

# initialize the ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever], weights=[0.5, 0.5]
)

# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
prompt = hub.pull("rlm/rag-prompt")

# 단계 7: 언어모델 생성(Create LLM)
# 모델(LLM) 을 생성합니다.
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)


def format_docs(docs):
    # 검색한 문서 결과를 하나의 문단으로 합쳐줍니다.
    return "\n\n".join(doc.page_content for doc in docs)


# 단계 8: 체인 생성(Create Chain)
rag_chain = (
    {"context": ensemble_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)



ensemble_retriever.get_relevant_documents('점심으로 뭘 먹었나요?')

/opt/anaconda3/envs/insta/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[Document(page_content='2019년10월28일 일기입니다. \n작가님의 보정과 내 보정 이렇게 보니 내가 뭘 더 보정해야하는지 알겠군! 수정해야겠다! 이 일기는 여민이가 나를 빡치게 한 일기~~*^^* 웩 단톡 갑분싸 다신 저기서 말 안하는 것으로 응징할래~~너무 가벼운 응징인가ㅠ'),
 Document(page_content='2016년5월17일 일기입니다. \n수민이가 정확하게 예상한 나의 청포도 주스!\n정말 근근히 살아가는 느낌이다\n무슨 정신으로 학교를 왔는지~\n이제 집가서 무슨 정신으로 있아야 하는지~~\n가서 보드를 타러 나가볼까 \n아니면 행렬이나 좀 할까...\n정신 사나와서 뭘 해야되는지 모르겠다'),
 Document(page_content='2021년6월1일 일기입니다. \n내가 받는 것보다는 내가 뭘 해줄 수 있는지 없는 지에 집중하기. 아무래도 내 그릇은 간장종지인듯 한데…보상같은거 바라지 말기 그런거 바랄 바에 그냥 아무것도 하지 말기 그냥 그런가보다 이 사람은 내가 아니니 내 마음과 같을 수 없음을 잊지말기 사랑을 받는 것보다는 내가 이 사람을 사랑하는 내 마음 자체에서 위안과 행복을 느끼길'),
 Document(page_content='2021년6월1일 일기입니다. \n어제-랩실 점심으로 교수님 서정원이랑 밥먹고 김용환과 인사하고 서진이와 밥먹은날-사람많이 만나서 좀 피곤한날? 그리고 현진을 일상에서 조금 빼고 내 일상으로 돌아간다고 생각하니 내 일상 너무 지루하고 재미없어서 지긋지긋했다…만나는 사람마다 요즘 재밌는일 없냐고 질문하기..갑작스럽게 ㄹㅇ 흑백된 조윤영 일상 생각해보면 이번주에 디펜스도 있고 목요일엔 면접도 있고 사진처럼 카겜도 면접가버려서…거기에 주말 내내 할이버지 미수잔치 할 생각하면 까마득한데 그럼에도 불구하고 겁나 지루함ㅋㅋㅋㅋ그래서 아 사람들을 좀 만나고 다니면 내 하루 덜 길고 덜지루하겠지란 생각과 사회성 다시 끌어올려야겠다는 생각으로 여러 약속잡아버림. 하영언랑 승훈띠랑 술약잡고 서현

In [29]:
vectorstore.save_local('./db/faiss')

In [14]:

print(ensemble_retriever.get_relevant_documents('워니랑 뭘했나요?'))
# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "워니랑 뭘했나요?'"
response = rag_chain.invoke(question)
print(response)

[Document(page_content='2023년10월20일 일기입니다. \n죽고싶었던 숙취를 이겨내고 점심산책🌿 세일러랑 워니랑 셋이 먹고 춥지만 멋진 산책을 즐겼다!'), Document(page_content='2022년10월19일 일기입니다. \n최강록 삭당 생각보다는 별루,,2차 술잡 맛잇엇어 재밋엇던 술자리 난 워니랑 술먹는거ㅜ조아 담엔 노애벙가자고 해봐야징~~~~~~대성공 졸려 죽어'), Document(page_content='2024년1월9일 일기입니다. \n어늘의 요약 이원쌤 가신다니@아쉬워 번따햇다ㅋㅋㅋㅋㅋ아저씨요..점심은 홍도식당에서 지미항 여진이랑 먹엇고 화계팀@바쁘다 우리회사@법카 다 앖어지나ㅋㅎ탕탕이 가지고와주시누잼을 소니랑 워니랑 다같이@나눠먹았고 탕탕의 파스티는@맛있었다! 눈이 펑펑 왔다 그리고 출근길에@들은 엉덩이@플레이리스트✊'), Document(page_content='2024년6월17일 일기입니다. \n일기쓰기귀찮다 항정살구우ㅝ먹음 워니랑 퇴근하면서 마약같이 생긴 약봉투 발견함 4-3에서 만나자고 누기 마약거래하는거 아니냐며 키득댐 현지 히서니한테 극대노함 운동 수영다녀오고 점심에 3.5키로 걷고 헬스도 다녀옴 점심은@도시락 내일도 도시락 나는 멋쟁이야'), Document(page_content='2023년2월17일 일기입니다. \n오딘 링크로 정신없던 하루 탕탕이 다음주부타 없다니 깜깜하다…사수가 없는 삶 제가 살아보겠습니다…점심은 륜경이랑 워니랑 온센 정신없이 일하다가 다시@저녁은 알콜루팡인데 생각보디 E가 많아서 내가 대화를 주도하지 않아도 돼서 너무@좋았다 술조 맛있고 안주도 믓있거 만원은 차고도 넘치게 뽑아서 만족만족 헨리랑 친해져서 만족. 나의 회사 생활이 좀 더 편해지길..🙏'), Document(page_content='2023년12월7일 일기입니다. \n기절하겟다\n\n정신차리고 다시 일기써야지! 그냥 너무 바빴다~~하지만 바쁜게 무료한거보다는 낫지. 드디어! 워니랑 회의를 드디어! 하게 되어서 전달

In [ ]:
# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "점심을 누구랑 먹었나요?"
response = rag_chain.invoke(question)
print(response)

In [ ]:
# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "워니가 뭘했나요?"
response = rag_chain.invoke(question)
print(response)

In [ ]:
# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "쏘워와 함께 했던 일들에 대해서 최대한 자세하게 요약해줘"
response = rag_chain.invoke(question)
print(response)

In [34]:
from langchain_core.prompts import ChatPromptTemplate


prompt_txt = """당신은 유능한 비서입니다. 유저들의 질문에 검색된 일기장 데이터를 기반으로 대답하세요. 만약 답을 모른다면, 그냥 모른다고 말하십시오. 단계별로 생각하고 답변을 작성해주세요. 
Question: {question} 
Context: {context} 
Answer:
"""

prompt=ChatPromptTemplate(input_variables=['context', 'question'],
                   metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'},
                   messages=[HumanMessagePromptTemplate(
                       prompt=PromptTemplate(input_variables=['context', 'question'], 
                                             template=prompt_txt))])

# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "쏘워와 함께 했던 일들에 대해서 말해줘"
response = rag_chain.invoke(question)
print(response)

NameError: name 'ChatPromptTemplate' is not defined

In [33]:
prompt = hub.pull("rlm/rag-prompt")
prompt
ChatPromptTemplate(input_variables=['context', 'question'],
                   metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'},
                   messages=[HumanMessagePromptTemplate(
                       prompt=PromptTemplate(input_variables=['context', 'question'], 
                                             template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"))])

ChatPromptTemplate(input_variables=['context', 'question'], metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"))])